In [1]:
# main.py

import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

import cv2
from ultralytics import YOLO
import numpy as np
import itertools
from tkinter import Tk
from tkinter.filedialog import askopenfilename
import sys
import pandas as pd
import time
from datetime import timedelta

# ---------- 選擇模型與影片 ----------
MODEL_PATH = "modelv1.pt"
Tk().withdraw()
VIDEO_PATH = askopenfilename(title="請選擇影片檔案", filetypes=[("MP4 files", "*.mp4"), ("All files", "*.*")])
if not VIDEO_PATH:
    print("未選擇影片，程式結束。")
    sys.exit()

# ---------- 畫選取區域 ----------
drawing = False
areas = []
temp = []

def draw_rect(event, x, y, flags, param):
    global drawing, temp, areas
    if event == cv2.EVENT_LBUTTONDOWN:
        drawing = True
        temp = [(x, y)]
    elif event == cv2.EVENT_LBUTTONUP:
        drawing = False
        temp.append((x, y))
        if len(temp) == 2:
            areas.append(tuple(temp))
            temp = []

cap = cv2.VideoCapture(VIDEO_PATH)
ret, first_frame = cap.read()
if not ret:
    print("無法讀取影片")
    sys.exit()

cv2.namedWindow("Draw Areas")
cv2.setMouseCallback("Draw Areas", draw_rect)
while True:
    disp = first_frame.copy()
    for rect in areas:
        cv2.rectangle(disp, rect[0], rect[1], (0, 255, 0), 2)
    if len(temp) == 2:
        cv2.rectangle(disp, temp[0], temp[1], (0, 0, 255), 2)
    cv2.imshow("Draw Areas", disp)
    key = cv2.waitKey(1) & 0xFF
    if key == 27:
        cap.release()
        cv2.destroyAllWindows()
        sys.exit()
    if key == ord('c'):
        if temp:
            temp.clear()
        elif areas:
            areas.pop()
    if key == 13 and len(areas) >= 1:
        break
cv2.destroyWindow("Draw Areas")

# ---------- 初始化 YOLO ----------
model = YOLO(MODEL_PATH)
names = model.names
palette = [(255,0,0),(0,255,0),(0,0,255),(255,255,0),(255,0,255),
           (0,255,255),(128,128,0),(128,0,128),(0,128,128),(255,255,255)]

# ---------- 初始化統計 ----------
counted_ids = {
    'motorcycle': set(),
    'car': set(),
    'truck': set(),
    'bus': set()
}
vehicle_counts = {
    'motorcycle': 0,
    'car': 0,
    'truck': 0,
    'bus': 0
}

crossing_log = []
segment_log = []

fps = cap.get(cv2.CAP_PROP_FPS)
frame_idx = 0
segment_seconds = 300
next_record_time = segment_seconds

# ---------- 判斷重疊 ----------
def is_overlap(box1, box2):
    ax1, ay1, ax2, ay2 = box1
    bx1, by1, bx2, by2 = box2
    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)
    return inter_x1 < inter_x2 and inter_y1 < inter_y2

# ---------- 主迴圈 ----------
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model.track(frame, persist=True, tracker="bytetrack.yaml")
    boxes = results[0].boxes

    if boxes is not None:
        ids = boxes.id.cpu().numpy() if boxes.id is not None else itertools.count()
        classes = boxes.cls.cpu().numpy()
        confs = boxes.conf.cpu().numpy()

        for box, tid, cls, conf in zip(boxes.xyxy.cpu().numpy(), ids, classes, confs):
            tid = int(tid)
            class_name = names[int(cls)]
            x1, y1, x2, y2 = map(int, box)
            det_box = (x1, y1, x2, y2)
            color = palette[int(cls) % len(palette)]

            label = f"{class_name} {conf:.2f}"
            cv2.rectangle(frame, (x1,y1), (x2,y2), color, 2)
            cv2.putText(frame, label, (x1, y1-8), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

            if class_name in vehicle_counts:
                for idx, area in enumerate(areas):
                    area_box = (
                        min(area[0][0], area[1][0]),
                        min(area[0][1], area[1][1]),
                        max(area[0][0], area[1][0]),
                        max(area[0][1], area[1][1])
                    )
                    if is_overlap(det_box, area_box) and tid not in counted_ids[class_name]:
                        vehicle_counts[class_name] += 1
                        counted_ids[class_name].add(tid)
                        time_sec = round(frame_idx / fps, 2)
                        time_str = str(timedelta(seconds=int(time_sec)))
                        crossing_log.append({
                            "id": tid,
                            "class": class_name,
                            "x": (x1 + x2) // 2,
                            "y": (y1 + y2) // 2,
                            "time_sec": time_sec,
                            "time_hms": time_str,
                            "area_index": idx
                        })
                        break

    # 畫區域
    for rect in areas:
        cv2.rectangle(frame, rect[0], rect[1], (0, 255, 0), 2)

    # 顯示結果
    y_offset = 40
    for i, cls_name in enumerate(['motorcycle', 'car', 'truck', 'bus']):
        count = vehicle_counts.get(cls_name, 0)
        cv2.putText(frame, f"{cls_name.capitalize()} passed: {count}",
                    (30, y_offset + i * 40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 255), 3)

    # 每 5 分鐘紀錄一次並重置
    frame_idx += 1
    current_time = frame_idx / fps
    if current_time >= next_record_time:
        log_row = {"minute": round(current_time / 60, 2)}
        log_row.update(vehicle_counts)
        segment_log.append(log_row)

        for k in vehicle_counts:
            vehicle_counts[k] = 0
        for k in counted_ids:
            counted_ids[k].clear()
        next_record_time += segment_seconds

    cv2.imshow("YOLOv8 Crossing Counter", frame)
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

# ---------- 儲存結果到 CSV ----------
df_crossing = pd.DataFrame(crossing_log)
df_crossing.to_csv("vehicle_crossing_result.csv", index=False, encoding="utf-8-sig")
print("已輸出 CSV：vehicle_crossing_result.csv")

df_summary = pd.DataFrame(segment_log)
df_summary.to_csv("vehicle_summary_5min.csv", index=False, encoding="utf-8-sig")
print("已輸出 CSV：vehicle_summary_5min.csv")


2025-07-08 15:09:12.298 python[13196:291783] The class 'NSOpenPanel' overrides the method identifier.  This method is implemented by class 'NSWindow'


FileNotFoundError: [Errno 2] No such file or directory: 'modelv1.pt'